# Chunking Strategy Benchmark — Institutional Decision Records

This notebook benchmarks chunking strategies for a **decision-record** RAG corpus
(the relational shape is `decision` 1—N `strategies`, `constraints`, `decisions`, `outcomes`).

Unlike the flat policy documents in the earlier experiment, each record here is short,
highly structured, and repeats the same skeleton (`title -> strategy -> constraints ->
decision -> outcome`) with only the specific wording differing per record. That shape
changes which chunking strategies make sense — see the discussion cells before each
strategy for why it is included and what it trades off.

Strategies benchmarked:
1. **Whole-Decision** — one chunk per decision record (full context, low precision)
2. **Fixed-Size (RecursiveCharacterTextSplitter)** — naive baseline, chunk_size/overlap on the concatenated record
3. **Field/Section Chunking** — one chunk per field (Strategy / Constraints / Decision / Outcome), since that's the actual relational grain
4. **Sentence Chunking** — each field split into individual sentences — the most granular option, suited to short, atomic institutional statements
5. **Semantic Chunking** — embedding-similarity based splits on the concatenated record

Embedding models compared: `all-MiniLM-L6-v2`, `BAAI/bge-small-en-v1.5`, `nomic-ai/nomic-embed-text-v1.5`.


In [1]:
# !pip install langchain-chroma


In [2]:
# !pip install langchain-experimental


In [3]:
# !pip install langchain-huggingface langchain-text-splitters sentence-transformers


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
import shutil
import os
import re
import json
import pandas as pd


## Load the data

The mock export is a **flat list** of decision records (this is what a
`SELECT ... FROM decision d JOIN strategies s ... JOIN constraints c ... JOIN outcomes o ...`
denormalized view would look like). Each record:

```json
{
  "id": "decision_001",
  "title": "Initiate Five-Year Program Review for BBA",
  "strategies": ["..."],
  "constraints": ["...", "..."],
  "decisions": ["..."],
  "outcomes": ["..."]
}
```

`title` is the only field that stays literally identical across a record's own strategy/
constraint/decision/outcome rows (they all belong to the same decision) — the actual
*content* that differs record-to-record is the problem-specific text inside each field.
That's why whole-record chunks end up dominated by repeated title/context language, and
why field- or sentence-level chunking (which isolates the differing content) is worth
testing against it.


In [5]:
with open("institutional_data (1).json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(len(data))
data[0]


<class 'list'>
33


{'id': 'decision_001',
 'title': 'Initiate Five-Year Program Review for BBA',
 'strategies': ['Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison. Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.'],
 'constraints': ['Changes require Academic Council approval at least one semester before implementation.',
  'All revisions must be reflected in the updated course catalog submitted to the affiliating university.'],
 'decisions': ['Authorize the immediate execution of a comprehensive curriculum review for the BBA program to align learning modules with regional commercial practices and national university guidelines.'],
 'outcomes': ['Program review completed and approved; updated BBA curriculum submitted to Academic Council.']}

## Ground truth Q&A set

Hand-written questions against a representative subset of decisions, covering the
different field types (constraint/time, role/approver, and outcome questions) so the
benchmark reflects real retrieval needs rather than just lexical overlap.


In [6]:
ground_truth = [

# decision_001 — Initiate Five-Year Program Review for BBA
{
    "question": "Who must approve curriculum changes before they are implemented?",
    "ground_truth_answer": "Academic Council",
    "decision_id": "decision_001",
    "answer_type": "Role"
},
{
    "question": "What was the outcome of the BBA five-year program review?",
    "ground_truth_answer": "Program review completed and approved; updated BBA curriculum submitted to Academic Council.",
    "decision_id": "decision_001",
    "answer_type": "Outcome"
},

# decision_003 — Adjudicate Academic Misconduct in Final Year Project
{
    "question": "How many working days do academic misconduct investigations have to be completed within?",
    "ground_truth_answer": "21 working days",
    "decision_id": "decision_003",
    "answer_type": "Time"
},
{
    "question": "What penalty was given to the students found guilty of plagiarism?",
    "ground_truth_answer": "A grade of zero was assigned for the project component.",
    "decision_id": "decision_003",
    "answer_type": "Outcome"
},

# decision_005 — Approve MOU with Commercial Bank for Internship Partnership
{
    "question": "Who signs the MOU with the commercial bank for the internship partnership?",
    "ground_truth_answer": "College Principal",
    "decision_id": "decision_005",
    "answer_type": "Role"
},
{
    "question": "How often must MOUs be reviewed and renewed?",
    "ground_truth_answer": "Every three years",
    "decision_id": "decision_005",
    "answer_type": "Constraint"
},

# decision_009 — Respond to Suspected Data Breach of Student Records
{
    "question": "Within how many hours must a data breach be escalated?",
    "ground_truth_answer": "72 hours",
    "decision_id": "decision_009",
    "answer_type": "Time"
},
{
    "question": "How many student records were identified as potentially exposed in the breach?",
    "ground_truth_answer": "120 student records",
    "decision_id": "decision_009",
    "answer_type": "Outcome"
},

# decision_011 — Contain and Recover from Ransomware Attack
{
    "question": "Within how many hours must a Level 1 critical incident be contained?",
    "ground_truth_answer": "4 hours",
    "decision_id": "decision_011",
    "answer_type": "Constraint"
},
{
    "question": "How long did it actually take to contain the ransomware attack?",
    "ground_truth_answer": "3 hours",
    "decision_id": "decision_011",
    "answer_type": "Outcome"
},

# decision_017 — Approve Faculty Sabbatical for Doctoral Research
{
    "question": "What is the maximum percentage of the departmental annual budget that CPD spending can reach?",
    "ground_truth_answer": "5%",
    "decision_id": "decision_017",
    "answer_type": "Constraint"
},
{
    "question": "How long was the sabbatical approved for in the doctoral research case?",
    "ground_truth_answer": "Two years",
    "decision_id": "decision_017",
    "answer_type": "Outcome"
},

# decision_020 — Approve Graduate Thesis Ethics Clearance
{
    "question": "Within how many days of thesis registration must ethics proposals be submitted?",
    "ground_truth_answer": "60 days",
    "decision_id": "decision_020",
    "answer_type": "Time"
},
{
    "question": "What was the final outcome of the graduate thesis ethics review?",
    "ground_truth_answer": "Full clearance was issued after the student revised the anonymization procedures.",
    "decision_id": "decision_020",
    "answer_type": "Outcome"
},

# decision_029 — Terminate Defective Non-Compliant ERP Software Contract
{
    "question": "How much was the retained execution bond after the ERP contract was cancelled?",
    "ground_truth_answer": "NPR 400,000",
    "decision_id": "decision_029",
    "answer_type": "Outcome"
},
{
    "question": "Within how many days must procurement evaluation reports be submitted to the Public Procurement Monitoring Office?",
    "ground_truth_answer": "30 days",
    "decision_id": "decision_029",
    "answer_type": "Constraint"
},

# decision_033 — Resolve Formal Academic Appeal on Final Grade Dispute
{
    "question": "Within how many days must an academic appeal be submitted after the triggering event?",
    "ground_truth_answer": "14 days",
    "decision_id": "decision_033",
    "answer_type": "Time"
},
{
    "question": "What was the outcome of the final grade dispute appeal?",
    "ground_truth_answer": "The appeal was upheld and the grade was revised from Fail to Pass.",
    "decision_id": "decision_033",
    "answer_type": "Outcome"
},
]

print(len(ground_truth))


18


In [7]:
embedding_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-small-en-v1.5",
    "Nomic": "nomic-ai/nomic-embed-text-v1.5"
}


In [8]:
def benchmark_embeddings(documents,
                          strategy_name,
                          persist_directory="vectorstores"):
    """
    documents:
        Output of your chunking function — list of dicts with
        "id", "text", "metadata".

    strategy_name:
        Example:
            WholeDecision
            FixedChunk
            FieldChunk
            SentenceChunk
            SemanticChunk
    """

    results = {}

    docs = [
        Document(
            page_content=d["text"],
            metadata=d["metadata"]
        )
        for d in documents
    ]

    for model_name, model in embedding_models.items():

        print("=" * 60)
        print(f"Chunking : {strategy_name}")
        print(f"Embedding: {model_name}")
        print("=" * 60)

        embedding = HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs={
                "trust_remote_code": True
            }
        )

        db_path = os.path.join(
            persist_directory,
            f"{strategy_name}_{model_name}"
        )

        if os.path.exists(db_path):
            shutil.rmtree(db_path)

        vectordb = Chroma.from_documents(
            docs,
            embedding,
            persist_directory=db_path
        )

        retriever = vectordb.as_retriever(
            search_kwargs={
                "k": 3
            }
        )

        results[model_name] = retriever

        print("Done.\n")

    return results


In [9]:
def evaluate_retrievers(retrievers, strategy_name):
    """
    Runs the ground_truth set against every model's retriever for a given
    chunking strategy and returns a results DataFrame with Hit@1 and Hit@3.
    """

    rows = []

    for model_name, retriever in retrievers.items():

        for sample in ground_truth:

            question = sample["question"]
            expected = sample["decision_id"]

            docs = retriever.invoke(question)
            retrieved_ids = [d.metadata.get("decision_id") for d in docs]

            hit_at_1 = retrieved_ids[0] == expected if retrieved_ids else False
            hit_at_3 = expected in retrieved_ids

            rows.append({
                "Strategy": strategy_name,
                "Embedding": model_name,
                "Question": question,
                "Expected": expected,
                "Top1_Retrieved": retrieved_ids[0] if retrieved_ids else None,
                "Hit@1": hit_at_1,
                "Hit@3": hit_at_3
            })

    return pd.DataFrame(rows)


def summarize(results_df):
    summary = (
        results_df
        .groupby(["Strategy", "Embedding"])[["Hit@1", "Hit@3"]]
        .mean()
        .round(3) * 100
    )
    summary.columns = ["Hit@1 (%)", "Hit@3 (%)"]
    return summary


all_results = []  # collects one DataFrame per strategy for the final comparison


## Strategy 1 — Whole-Decision Chunking

One chunk per decision record: `title + strategy + constraints + decision + outcome`
concatenated together.

**Why include it:** it's the simplest baseline and guarantees the model always sees the
full context of a decision (no cross-field information is ever split apart).

**Trade-off to watch:** because `title` and the general phrasing repeat across a
decision's own fields, and because full records run long, embeddings get "diluted" —
a question about one specific constraint has to compete with the semantics of the whole
record, which tends to hurt precision on narrow factual questions (exactly the "Time" /
"Constraint" questions in the ground truth set).


In [10]:
def build_decision_text(rec):
    parts = [f"Decision: {rec['title']}"]

    if rec.get("strategies"):
        parts.append("Strategy: " + " ".join(rec["strategies"]))
    if rec.get("constraints"):
        parts.append("Constraints: " + " ".join(rec["constraints"]))
    if rec.get("decisions"):
        parts.append("Decision Taken: " + " ".join(rec["decisions"]))
    if rec.get("outcomes"):
        parts.append("Outcome: " + " ".join(rec["outcomes"]))

    return "\n\n".join(parts)


def whole_decision_chunking(data):

    documents = []

    for rec in data:

        documents.append({
            "id": rec["id"],
            "text": build_decision_text(rec),
            "metadata": {
                "decision_id": rec["id"],
                "title": rec["title"]
            }
        })

    return documents


documents = whole_decision_chunking(data)

print(len(documents))
print(documents[0]["text"])


33
Decision: Initiate Five-Year Program Review for BBA

Strategy: Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison. Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.

Constraints: Changes require Academic Council approval at least one semester before implementation. All revisions must be reflected in the updated course catalog submitted to the affiliating university.

Decision Taken: Authorize the immediate execution of a comprehensive curriculum review for the BBA program to align learning modules with regional commercial practices and national university guidelines.

Outcome: Program review completed and approved; updated BBA curriculum submitted to Academic Council.


In [11]:
whole_decision_retrievers = benchmark_embeddings(
    documents,
    "WholeDecision"
)


Chunking : WholeDecision
Embedding: MiniLM
Done.

Chunking : WholeDecision
Embedding: BGE
Done.

Chunking : WholeDecision
Embedding: Nomic


<All keys matched successfully>


Done.



In [12]:
whole_decision_results = evaluate_retrievers(whole_decision_retrievers, "WholeDecision")
all_results.append(whole_decision_results)

summarize(whole_decision_results)


Hit@1 (%)  Hit@3 (%)
Strategy      Embedding                      
WholeDecision BGE             88.9      100.0
              MiniLM          88.9      100.0
              Nomic           83.3      100.0

## Strategy 2 — Fixed-Size Chunking (RecursiveCharacterTextSplitter)

Splits the same concatenated record text on a fixed character budget, ignoring field
boundaries.

**Why include it:** it's the standard naive baseline in most RAG tutorials — useful to
confirm it underperforms structure-aware strategies on structured data like this, rather
than just assuming it.

**Trade-off to watch:** because most records here are short (well under the chunk_size),
this strategy will frequently collapse back down to ~1 chunk per record, behaving almost
identically to Whole-Decision chunking — the difference only shows up on the handful of
longer records, and where it does, it risks slicing a sentence (e.g. a constraint) in
half across two chunks.


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def fixed_chunking(data, chunk_size=400, chunk_overlap=60):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    documents = []

    for rec in data:

        text = build_decision_text(rec)
        chunks = splitter.split_text(text)

        for i, chunk in enumerate(chunks):

            documents.append({
                "id": f'{rec["id"]}_{i}',
                "text": chunk,
                "metadata": {
                    "decision_id": rec["id"],
                    "title": rec["title"]
                }
            })

    return documents


documents = fixed_chunking(data)

print(len(documents))
print(documents[:2])


99
[{'id': 'decision_001_0', 'text': 'Decision: Initiate Five-Year Program Review for BBA\n\nStrategy: Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison. Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA'}}, {'id': 'decision_001_1', 'text': 'Constraints: Changes require Academic Council approval at least one semester before implementation. All revisions must be reflected in the updated course catalog submitted to the affiliating university.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA'}}]


In [14]:
fixed_retrievers = benchmark_embeddings(
    documents,
    "FixedChunk"
)


Chunking : FixedChunk
Embedding: MiniLM
Done.

Chunking : FixedChunk
Embedding: BGE
Done.

Chunking : FixedChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [15]:
fixed_results = evaluate_retrievers(fixed_retrievers, "FixedChunk")
all_results.append(fixed_results)

summarize(fixed_results)


Hit@1 (%)  Hit@3 (%)
Strategy   Embedding                      
FixedChunk BGE             94.4      100.0
           MiniLM          94.4      100.0
           Nomic           94.4      100.0

## Strategy 3 — Field/Section Chunking

One chunk per **field** (Strategy, Constraints, Decision Taken, Outcome), each prefixed
with the decision title for context, stored with a `section` metadata tag.

**Why include it:** this mirrors the actual relational grain of the source data
(`decision` -> `strategies` / `constraints` / `decisions` / `outcomes` as separate
tables/rows). Because a constraint question should only need to match the Constraints
field, isolating fields removes the "dilution" problem from Strategy 1 while keeping
each chunk self-contained enough to be useful on its own (title is repeated in).

**Trade-off to watch:** a question that spans two fields (e.g. "what strategy led to
what outcome?") won't be answerable from a single retrieved chunk — you'd need to
retrieve k>1 and reason across chunks, or fall back to a parent-document lookup by
`decision_id`.


In [16]:
def field_chunking(data):

    documents = []

    field_map = [
        ("strategies", "Strategy"),
        ("constraints", "Constraints"),
        ("decisions", "Decision Taken"),
        ("outcomes", "Outcome"),
    ]

    for rec in data:

        for field_key, section_name in field_map:

            values = rec.get(field_key) or []

            if not values:
                continue

            text = f"Decision: {rec['title']}\n{section_name}: " + " ".join(values)

            documents.append({
                "id": f'{rec["id"]}_{section_name}',
                "text": text,
                "metadata": {
                    "decision_id": rec["id"],
                    "title": rec["title"],
                    "section": section_name
                }
            })

    return documents


documents = field_chunking(data)

print(len(documents))
print(documents[:4])


132
[{'id': 'decision_001_Strategy', 'text': 'Decision: Initiate Five-Year Program Review for BBA\nStrategy: Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison. Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA', 'section': 'Strategy'}}, {'id': 'decision_001_Constraints', 'text': 'Decision: Initiate Five-Year Program Review for BBA\nConstraints: Changes require Academic Council approval at least one semester before implementation. All revisions must be reflected in the updated course catalog submitted to the affiliating university.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA', 'section': 'Constraints'}}, {'id': 'decision_001_Decision Taken', 'text': 'Decision: Initiate Five-Year Program Review fo

In [17]:
field_retrievers = benchmark_embeddings(
    documents,
    "FieldChunk"
)


Chunking : FieldChunk
Embedding: MiniLM
Done.

Chunking : FieldChunk
Embedding: BGE
Done.

Chunking : FieldChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [18]:
field_results = evaluate_retrievers(field_retrievers, "FieldChunk")
all_results.append(field_results)

summarize(field_results)


Hit@1 (%)  Hit@3 (%)
Strategy   Embedding                      
FieldChunk BGE             88.9       94.4
           MiniLM          94.4      100.0
           Nomic           94.4       94.4

## Strategy 4 — Sentence Chunking

Each field is split further into **individual sentences**, so a constraint list item
that itself contains two sentences (e.g. "X must happen. Y must be documented.") becomes
two separate chunks. This is the strategy you specifically asked to test, and it fits
this dataset well because most field values are already short, atomic, single-topic
statements — splitting to sentence level maximizes retrieval precision without leaving
much useful context on the floor.

**Why include it:** institutional constraint/outcome text is dense with independent
facts packed into short paragraphs (a time limit, a role, a follow-up condition, all in
one list item). Sentence-level chunks let a narrow factual question (like the "Time"
questions in the ground truth set) match almost exactly, instead of competing against
neighboring facts inside the same chunk.

**Trade-off to watch:** very short chunks (sometimes only a few words once title
context is stripped) can lose disambiguating context, and you get many more vectors to
store/search — for a 33-record dataset this is trivial, but it's the axis to watch if
this pipeline is pointed at the full production database.


In [19]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(])")


def split_sentences(text):
    text = text.strip()
    if not text:
        return []
    sentences = SENTENCE_SPLIT_RE.split(text)
    return [s.strip() for s in sentences if s.strip()]


def sentence_chunking(data):

    documents = []

    field_map = [
        ("strategies", "Strategy"),
        ("constraints", "Constraints"),
        ("decisions", "Decision Taken"),
        ("outcomes", "Outcome"),
    ]

    for rec in data:

        for field_key, section_name in field_map:

            values = rec.get(field_key) or []

            sent_idx = 0

            for value in values:
                for sentence in split_sentences(value):

                    text = f"Decision: {rec['title']}\n{section_name}: {sentence}"

                    documents.append({
                        "id": f'{rec["id"]}_{section_name}_{sent_idx}',
                        "text": text,
                        "metadata": {
                            "decision_id": rec["id"],
                            "title": rec["title"],
                            "section": section_name
                        }
                    })

                    sent_idx += 1

    return documents


documents = sentence_chunking(data)

print(len(documents))
print(documents[:4])


302
[{'id': 'decision_001_Strategy_0', 'text': 'Decision: Initiate Five-Year Program Review for BBA\nStrategy: Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA', 'section': 'Strategy'}}, {'id': 'decision_001_Strategy_1', 'text': 'Decision: Initiate Five-Year Program Review for BBA\nStrategy: Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA', 'section': 'Strategy'}}, {'id': 'decision_001_Constraints_0', 'text': 'Decision: Initiate Five-Year Program Review for BBA\nConstraints: Changes require Academic Council approval at least one semester before implementation.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review

In [20]:
sentence_retrievers = benchmark_embeddings(
    documents,
    "SentenceChunk"
)


Chunking : SentenceChunk
Embedding: MiniLM
Done.

Chunking : SentenceChunk
Embedding: BGE
Done.

Chunking : SentenceChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [21]:
sentence_results = evaluate_retrievers(sentence_retrievers, "SentenceChunk")
all_results.append(sentence_results)

summarize(sentence_results)


Hit@1 (%)  Hit@3 (%)
Strategy      Embedding                      
SentenceChunk BGE             88.9      100.0
              MiniLM         100.0      100.0
              Nomic           94.4      100.0

## Strategy 5 — Semantic Chunking

Uses `SemanticChunker` (embedding-similarity breakpoints) on the same concatenated
record text from Strategy 1.

**Why include it:** it's the strategy most often recommended by default for RAG, so
it's worth benchmarking directly against the structure-aware options above rather than
assuming it wins.

**Trade-off to watch:** semantic breakpoint detection needs enough text to find a
similarity "cliff" — most records here are short enough that SemanticChunker will often
return the whole record as a single chunk anyway (behaving like Strategy 1), while
adding embedding-time cost per chunk decision. It tends to pay off more on longer,
free-flowing prose than on already-templated short records like these.


In [22]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

semantic_splitter = SemanticChunker(semantic_embedding)


def semantic_chunking(data):

    documents = []

    for rec in data:

        text = build_decision_text(rec)
        chunks = semantic_splitter.split_text(text)

        for i, chunk in enumerate(chunks):

            documents.append({
                "id": f'{rec["id"]}_{i}',
                "text": chunk,
                "metadata": {
                    "decision_id": rec["id"],
                    "title": rec["title"]
                }
            })

    return documents


documents = semantic_chunking(data)

print(len(documents))
print(documents[:2])


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4524\851182828.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


66
[{'id': 'decision_001_0', 'text': 'Decision: Initiate Five-Year Program Review for BBA\n\nStrategy: Form a review committee comprising the Department Head, two senior faculty, one industry representative, and the QA Office Liaison. Conduct gap analysis against UGC Program Approval Guidelines and current market needs over an eight-week period.', 'metadata': {'decision_id': 'decision_001', 'title': 'Initiate Five-Year Program Review for BBA'}}, {'id': 'decision_001_1', 'text': 'Constraints: Changes require Academic Council approval at least one semester before implementation. All revisions must be reflected in the updated course catalog submitted to the affiliating university. Decision Taken: Authorize the immediate execution of a comprehensive curriculum review for the BBA program to align learning modules with regional commercial practices and national university guidelines. Outcome: Program review completed and approved; updated BBA curriculum submitted to Academic Council.', 'meta

In [23]:
semantic_retrievers = benchmark_embeddings(
    documents,
    "SemanticChunk"
)


Chunking : SemanticChunk
Embedding: MiniLM
Done.

Chunking : SemanticChunk
Embedding: BGE
Done.

Chunking : SemanticChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [24]:
semantic_results = evaluate_retrievers(semantic_retrievers, "SemanticChunk")
all_results.append(semantic_results)

summarize(semantic_results)


Hit@1 (%)  Hit@3 (%)
Strategy      Embedding                      
SemanticChunk BGE             88.9      100.0
              MiniLM          94.4      100.0
              Nomic           88.9      100.0

## Final comparison — all strategies x all models

`Hit@1` = the top retrieved chunk belongs to the correct decision.
`Hit@3` = the correct decision appears anywhere in the top 3 retrieved chunks
(more forgiving, and the more realistic metric once field/sentence chunking means
several chunks can legitimately come from the same decision).


In [25]:
master_results = pd.concat(all_results, ignore_index=True)

final_summary = summarize(master_results)
final_summary


Hit@1 (%)  Hit@3 (%)
Strategy      Embedding                      
FieldChunk    BGE             88.9       94.4
              MiniLM          94.4      100.0
              Nomic           94.4       94.4
FixedChunk    BGE             94.4      100.0
              MiniLM          94.4      100.0
              Nomic           94.4      100.0
SemanticChunk BGE             88.9      100.0
              MiniLM          94.4      100.0
              Nomic           88.9      100.0
SentenceChunk BGE             88.9      100.0
              MiniLM         100.0      100.0
              Nomic           94.4      100.0
WholeDecision BGE             88.9      100.0
              MiniLM          88.9      100.0
              Nomic           83.3      100.0

In [26]:
pivot_hit1 = final_summary["Hit@1 (%)"].unstack("Embedding")
pivot_hit3 = final_summary["Hit@3 (%)"].unstack("Embedding")

print("Hit@1 (%) by strategy x embedding model")
display(pivot_hit1)

print("\nHit@3 (%) by strategy x embedding model")
display(pivot_hit3)


Hit@1 (%) by strategy x embedding model


Embedding,BGE,MiniLM,Nomic
Strategy,,,
FieldChunk,88.9,94.4,94.4
FixedChunk,94.4,94.4,94.4
SemanticChunk,88.9,94.4,88.9
SentenceChunk,88.9,100.0,94.4
WholeDecision,88.9,88.9,83.3



Hit@3 (%) by strategy x embedding model


Embedding,BGE,MiniLM,Nomic
Strategy,,,
FieldChunk,94.4,100.0,94.4
FixedChunk,100.0,100.0,100.0
SemanticChunk,100.0,100.0,100.0
SentenceChunk,100.0,100.0,100.0
WholeDecision,100.0,100.0,100.0


## Reading the results / recommendation

Fill in after running, but expect (and verify against your actual numbers):

- **Field or Sentence chunking should win on Hit@1** for narrow factual questions
  (Time/Constraint answer types), because they isolate the answer-bearing text instead
  of burying it inside a full record.
- **Whole-Decision chunking should win on Hit@3-vs-Hit@1 gap being small** (i.e. it's
  consistently "roughly right" since there's only one chunk per decision to begin with),
  but usually loses outright accuracy versus Field/Sentence chunking.
- **Fixed-size chunking** should track Whole-Decision closely on this dataset, since
  most records are short enough not to be split — it earns its keep once your production
  database has longer free-text fields (e.g. long-form `description` columns) rather
  than the short bullet-style strategy/constraint/outcome text in this mock set.
- **Semantic chunking**'s value proposition — variable-length, topic-coherent chunks —
  matters most on longer, less-templated prose. On this templated, already-short
  dataset it mostly collapses to Whole-Decision behavior at extra compute cost, so it's
  worth re-testing once your real tables include longer narrative fields (e.g. a full
  incident write-up or committee minutes) rather than being adopted by default.

**Practical recommendation for this schema:** use **Field or Sentence chunking as the
retrieval index**, always carrying `decision_id` in metadata, and pair it with a
**parent-document / decision-level lookup** at answer time — retrieve the precise
field/sentence chunk for ranking, then optionally pull the full decision record (all
four fields) by `decision_id` before generating the final answer. That gets you the
precision of narrow chunks and the full context of whole-record chunks without forcing
a single strategy to do both jobs.

For the **embedding model**, compare the `Hit@1`/`Hit@3` columns above per model across
strategies (not just the best strategy) — a model that's consistently mid-table across
every strategy is a safer production choice than one that only wins on its best-case
strategy, since your production data won't map perfectly onto any one synthetic
benchmark.
